# Lesson 6.1 - KNN

**Objectives**

By the end of this lesson, you will be able to:

- Prepare a real-world dataset for classification: clean it and turn CO2 emissions growth into increasing/decreasing labels
- Train a K-Nearest Neighbors (KNN) classifier with scikit-learn and evaluate it with test accuracy
- Tune KNN hyperparameters (`n_neighbors`, `weights`) by comparing test accuracy across values of `k`
- Explain why KNN needs feature scaling, and compare models trained with and without `StandardScaler`

## 1. Import python packages: matplotlib, pandas, numpy

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 2. Read CO2 emission data from csv file

In [ ]:
data_file_path = "../../data/owid-co2-data.csv"
co2_emission = pd.read_csv(data_file_path, sep=",")

## 3. Prepare Dataset

In the dataset, there are many rows with country name of "Africa", "Asia", "World"
Which are not really belong to "Country", those entries do not have iso_code, so
we can filter them out by `iso_code is NULL` and `empty`.

In [ ]:
# Filter out any non-country entries in the CO2 emissions dataset
# Aggregates like "Africa", "Asia", "World" have an empty iso_code; real countries have one.
countries = co2_emission.loc[
    co2_emission["iso_code"].notna() & (co2_emission["iso_code"] != "")
]

Check how many countries we have:

In [ ]:
countries["country"].unique()

In [ ]:
# Select CO2 emissions data for the year 2022 and related columns:

# - country: Country name
# - year: year of the observation
# - population: total population of the country
# - gdp: Gross Domestic Product of the country, international-$
# - co2: annual co2 emissions, Mt (million tonnes)
# - co2_per_capita: amount of CO2 a country emits per person in one year, t/person
# - co2_growth_prct: annual CO₂ emissions growth (%), annual percentage growth in total emissions of carbon dioxide (CO₂), excluding land-use change

df_co2_2022 = countries.loc[
    countries["year"] == 2022,
    [
        "country",
        "year",
        "population",
        "gdp",
        "co2",
        "co2_per_capita",
        "co2_growth_prct",
    ],
]

In [ ]:
df_co2_2022.head(10)

Check if the poplation, gdb, co2, co2_per_capita, co2_growth_prct has NULL/NAN values:

In [ ]:
df_co2_2022.loc[df_co2_2022["population"].isna()]

In [ ]:
df_co2_2022.loc[df_co2_2022["gdp"].isna()].shape

In [ ]:
df_co2_2022.loc[df_co2_2022["co2"].isna()]

In [ ]:
df_co2_2022.loc[df_co2_2022["co2_per_capita"].isna()]

In [ ]:
df_co2_2022.loc[df_co2_2022["co2_growth_prct"].isna()]

Filter out those NULL/NAN from the dataset:

In [ ]:
df_co2_2022_cleaned = df_co2_2022.dropna(
    subset=["population", "gdp", "co2", "co2_per_capita", "co2_growth_prct"]
)

In [ ]:
df_co2_2022_cleaned.head(10)

In [ ]:
df_co2_2022_cleaned.shape

Reset the index so rows are numbered 1, 2, 3, ... instead of the original row numbers

In [ ]:
df_co2_2022_cleaned = df_co2_2022_cleaned.reset_index(drop=True)
df_co2_2022_cleaned.index = df_co2_2022_cleaned.index + 1

In [ ]:
# - a total of 218 rows,
# - 54 rows (NULL/NAN) removed in the cleaned dataset
# - the cleaned dataset has a total of 164 rows (countries) for 2022 years' co2 emissions data
df_co2_2022_cleaned

## 4. Assign co2 growth label to the dataset

- Classified data by co2 growth percentage in 2022:
- Negative grwoth (Decreasing): co2_growth_prct <= 0.0
  - `co2_growth_target = 0`
  - `co2_growth_label = "decreasing"`
- Positive growth (Increasing): co2_growth_prct > 0.0
  - `co2_growth_label = 1`
  - `co2_growth_label = "increasing"`

In [ ]:
df_co2_2022_labeled = df_co2_2022_cleaned.copy()
df_co2_2022_labeled["co2_growth_target"] = np.where(
    df_co2_2022_labeled["co2_growth_prct"] > 0, 1, 0
)

df_co2_2022_labeled["co2_growth_label"] = np.where(
    df_co2_2022_labeled["co2_growth_prct"] > 0, "increasing", "decreasing"
)

In [ ]:
df_co2_2022_labeled.head(10)

## 5. Understand how the data looks like using scatter plots

- [Main reference for matplotlib point styling](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.scatter.html)
- [Color names you can use](https://matplotlib.org/stable/gallery/color/named_colors.html)
- [Marker shapes](https://matplotlib.org/stable/api/markers_api.html)
- [Color maps](https://matplotlib.org/stable/users/explain/colors/colormaps.html)

In [ ]:
ax = df_co2_2022_labeled.plot.scatter(
    x="gdp",
    y="co2_per_capita",
    title="CO2 Emissions per Capita vs GDP in 2022",
    logx=True,
    label="CO2 per Capita",
    grid=True,
    xlabel="GDP",
    ylabel="CO2 per Capita (t/person)",
    marker="o",  # shape of the points
    c="red",  # color of the points
    s=30,  # size of the points
    edgecolor="grey",  # edge color of the points
    linewidth=0.6,  # edge line width of the points
    alpha=0.8,  # transparency of the points
)

ax.figure.set_facecolor("white")  # the whole figure (outside the axes)
ax.set_facecolor("#f2f2f2")  # the plotting area (inside the axes)

ax.grid(color="white", linewidth=1)
ax.set_axisbelow(True)

plt.show()

In [ ]:
ax = df_co2_2022_labeled.plot.scatter(
    x="gdp",
    y="co2_growth_prct",
    title="CO2 Emissions Growth vs GDP in 2022",
    logx=True,
    label="CO2 Growth",
    grid=True,
    xlabel="GDP",
    ylabel="CO2 Growth (%)",
    marker="o",  # shape of the points
    c="red",  # color of the points
    s=30,  # size of the points
    edgecolor="grey",  # edge color of the points
    linewidth=0.6,  # edge line width of the points
    alpha=0.8,  # transparency of the points
)

ax.figure.set_facecolor("white")  # the whole figure (outside the axes)
ax.set_facecolor("#f2f2f2")  # the plotting area (inside the axes)

ax.grid(color="white", linewidth=1)
ax.set_axisbelow(True)

plt.show()

In [ ]:
ax = df_co2_2022_labeled.plot.scatter(
    x="co2_per_capita",
    y="co2_growth_prct",
    title="CO2 Emissions Growth vs CO2 per Capita in 2022",
    logx=True,
    grid=True,
    xlabel="CO2 per Capita (t/person)",
    ylabel="CO2 Growth (%)",
    marker="o",  # shape of the points
    c="red",  # color of the points
    s=30,  # size of the points
    edgecolor="grey",  # edge color of the points
    linewidth=0.6,  # edge line width of the points
    alpha=0.8,  # transparency of the points
)

ax.figure.set_facecolor("white")  # the whole figure (outside the axes)
ax.set_facecolor("#f2f2f2")  # the plotting area (inside the axes)

ax.grid(color="white", linewidth=1)
ax.set_axisbelow(True)

plt.show()

In [ ]:
from matplotlib.lines import Line2D

# One color per co2_growth_label, used for both the points and the legend
label_colors = {"decreasing": "tab:green", "increasing": "tab:red"}

ax = df_co2_2022_labeled.plot.scatter(
    x="gdp",
    y="co2_per_capita",
    title="CO2 Emissions per Capita vs GDP in 2022",
    logx=True,
    logy=True,
    grid=True,
    xlabel="GDP",
    ylabel="CO2 per Capita (t/person)",
    marker="o",  # shape of the points
    c=df_co2_2022_labeled["co2_growth_label"].map(label_colors),  # color of the points
    s=30,  # size of the points
    edgecolor="grey",  # edge color of the points
    linewidth=0.6,  # edge line width of the points
    alpha=0.8,  # transparency of the points
)

# Legend: one marker per label, drawn in the same color as the points
legend_handles = [
    Line2D(
        [],
        [],
        marker="o",
        linestyle="",
        markerfacecolor=color,
        markeredgecolor="grey",
        markersize=7,
        label=label,
    )
    for label, color in label_colors.items()
]
ax.legend(handles=legend_handles, title="CO2 growth")

ax.figure.set_facecolor("white")  # the whole figure (outside the axes)
ax.set_facecolor("#f2f2f2")  # the plotting area (inside the axes)

ax.grid(color="white", linewidth=1)
ax.set_axisbelow(True)

plt.show()

## 6. Build a KNN model

### 6.1 Define a problem:

Predict co2 emission growth is increasing or decreasing for each country based on its gdp and co2 emission data per person in that year.

Input:

- gdp
- co2_per_capita

Output:

- co2_growth_target: increasing (`target = 1`), decreasing (`target = 0`)

In [ ]:
df_co2_2022_labeled.head(10)

In [ ]:
df_model_data = df_co2_2022_labeled.copy().loc[
    :, ["gdp", "co2_per_capita", "co2_growth_target"]
]

In [ ]:
df_model_data.head()

### 6.2 Split dataset into Training and Testing

In [ ]:
from sklearn.model_selection import train_test_split

X = df_model_data.loc[:, ["gdp", "co2_per_capita"]]
y = df_model_data["co2_growth_target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=0)

In [ ]:
### Train-Test Split Summary
print("### Train-Test Split Summary")
print(f"Overall dataset shape: {X.shape}")
print(
    f"Training set shape: {X_train.shape}, percentage: {X_train.shape[0] / X.shape[0]:.2%}"
)
print(f"Test set shape: {X_test.shape}, percentage: {X_test.shape[0] / X.shape[0]:.2%}")

### 6.3 Build the KNN classifier

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier

estimator = KNeighborsClassifier(
    n_neighbors=9,
    weights="uniform",  # or "distance"
    algorithm="auto",  # or "ball_tree", "kd_tree", "brute"
)

estimator.fit(X_train, y_train)
y_pred = estimator.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

In [ ]:
y_pred_df = pd.DataFrame(
    {"y_test": y_test, "y_pred": y_pred, "correct": y_test == y_pred}
)

In [ ]:
y_pred_df.head(10)
# y_pred_df.loc[y_pred_df["correct"]].shape[0] * 100/ y_pred_df.shape[0]

### 6.4 Model's Hyper-parameters tuning

In [ ]:
metrics = pd.DataFrame(
    {
        "n_neighbors": [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23],
        "uniform_accuracy": [None] * 12,
        "distance_accuracy": [None] * 12,
    }
)
metrics.head()

In [ ]:
for weights in ["uniform", "distance"]:
    print(f"##### Training KNN with weights={weights}")
    for i, n_neighbors in enumerate(metrics["n_neighbors"]):
        estimator = KNeighborsClassifier(
            n_neighbors=n_neighbors,
            weights=weights,  # or "distance"
            algorithm="auto",  # or "ball_tree", "kd_tree", "brute"
        )

        estimator.fit(X_train, y_train)
        y_pred = estimator.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        if weights == "uniform":
            metrics.loc[i, "uniform_accuracy"] = accuracy
        else:
            metrics.loc[i, "distance_accuracy"] = accuracy
        print(
            f"Training KNN with n_neighbors={n_neighbors} and weights={weights}, accuracy={accuracy}"
        )


In [ ]:
metrics

In [ ]:
fig, ax = plt.subplots()

# The accuracy columns started as None, so convert them to numbers before plotting
ax.plot(
    metrics["n_neighbors"],
    metrics["uniform_accuracy"].astype(float),
    marker="o",
    color="tab:blue",
    label="uniform",
)
ax.plot(
    metrics["n_neighbors"],
    metrics["distance_accuracy"].astype(float),
    marker="o",
    color="tab:red",
    label="distance",
)

ax.set_title("KNN Test Accuracy vs Number of Neighbors")
ax.set_xlabel("n_neighbors (k)")
ax.set_ylabel("Test accuracy")
ax.set_xticks(metrics["n_neighbors"])
ax.legend(title="weights")

ax.figure.set_facecolor("white")  # the whole figure (outside the axes)
ax.set_facecolor("#f2f2f2")  # the plotting area (inside the axes)

ax.grid(color="white", linewidth=1)
ax.set_axisbelow(True)

plt.show()

### 6.5 Why applying scaler to the input dataset?

In [ ]:
metrics_scaling = pd.DataFrame(
    {
        "n_neighbors": [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23],
        "without_scaling": [None] * 12,
        "with_scaling": [None] * 12,
    }
)
metrics_scaling.head()

In [ ]:
# apply StandardScaler to the features before fitting the KNN model

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
for model in ["without_scaling", "with_scaling"]:
    print(f"##### Training KNN with model={model}")
    for i, n_neighbors in enumerate(metrics_scaling["n_neighbors"]):
        estimator = KNeighborsClassifier(
            n_neighbors=n_neighbors,
            weights="uniform",
            algorithm="auto",
        )

        if model == "with_scaling":
            estimator.fit(X_train_scaled, y_train)
            y_pred = estimator.predict(X_test_scaled)
        else:
            estimator.fit(X_train, y_train)
            y_pred = estimator.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)
        if model == "without_scaling":
            metrics_scaling.loc[i, "without_scaling"] = accuracy
        else:
            metrics_scaling.loc[i, "with_scaling"] = accuracy
        print(
            f"Training KNN with n_neighbors={n_neighbors} and model={model}, accuracy={accuracy}"
        )

In [ ]:
fig, ax = plt.subplots()

# The accuracy columns started as None, so convert them to numbers before plotting
ax.plot(
    metrics_scaling["n_neighbors"],
    metrics_scaling["without_scaling"].astype(float),
    marker="o",
    color="tab:blue",
    label="without_scaling",
)
ax.plot(
    metrics_scaling["n_neighbors"],
    metrics_scaling["with_scaling"].astype(float),
    marker="o",
    color="tab:red",
    label="with_scaling",
)

ax.set_title("KNN Test Accuracy vs Number of Neighbors")
ax.set_xlabel("n_neighbors (k)")
ax.set_ylabel("Test accuracy")
ax.set_xticks(metrics_scaling["n_neighbors"])
ax.legend(title="scaling")

ax.figure.set_facecolor("white")  # the whole figure (outside the axes)
ax.set_facecolor("#f2f2f2")  # the plotting area (inside the axes)

ax.grid(color="white", linewidth=1)
ax.set_axisbelow(True)

plt.show()

## 7. Save the cleaned datasets into file

In [ ]:
df_co2_2022_labeled.to_csv("../../data/df_co2_2022_labeled.csv", index=False)